In [1]:


def clean_text(text: str) -> str:
    """Remove newlines, tabs, extra spaces, punctuation and lowercase."""
    text = re.sub(r"[\r\n\t]+", " ", text)
    text = re.sub(r"\s+", " ", text)
    text = re.sub(r"[!?]+", "", text)
    return text.strip().lower()

## SQLite

In [2]:

import sqlite3

conn = sqlite3.connect(r"/src/triplet_extraction/batch_preprocessing/GTVT_law.db")
cursor = conn.cursor()

def extract_all_with_parents(cursor):
    # Step 1: Find all "Điểm" nodes
    cursor.execute("SELECT * FROM laws WHERE title LIKE '%Điểm%'")
    rows = cursor.fetchall()
    if not rows:
        return []

    columns = [col[0] for col in cursor.description]
    results = []

    # Step 2: For each node, find its parent chain
    for row in rows:
        node = dict(zip(columns, row))
        parent_chain = []

        current = node
        visited = set()

        while current['parent_id'] is not None and current['parent_id'] not in visited:
            visited.add(current['id'])
            cursor.execute("SELECT * FROM laws WHERE id = ?", (current['parent_id'],))
            parent = cursor.fetchone()
            if not parent:
                break
            parent_dict = dict(zip(columns, parent))
            parent_chain.append(parent_dict)
            current = parent_dict

        results.append({
            "target": node,
            "ancestors": parent_chain[::-1]
        })

    return results

In [4]:
diem_db_data = extract_all_with_parents(cursor)
diem_data = {}

for diem in diem_db_data:
    content = ""
    title = ""

    for r in diem["ancestors"]:
        title += r['title'] + " "
        if not (r['title'].strip().startswith("Chương") or r['title'].strip().startswith("Điều")):
            content += r['content'] + "\n"

    title += diem['target']["title"]
    content += diem['target']["content"]

    so_hieu = diem['target']['so_hieu']
    ID = diem['target']['id']

    diem_data[ID] = {
        "so_hieu": so_hieu,
        "title": title,
        "content": content
    }

## MongoDB

In [18]:
from pymongo import MongoClient
import re

def extract_all_with_parents(collection):
    # Step 1: Find all "Điểm" nodes
    diem_nodes = list(collection.find({"title": {"$regex": "điểm", "$options": "i"}}))

    if not diem_nodes:
        return []

    results = []

    # Step 2: For each node, find its parent chain
    for node in diem_nodes:
        parent_chain = []
        current = node
        visited = set()

        if node.get('is_phu_luc'):
            continue

        while current.get('parent_id') is not None and current.get('parent_id') not in visited:
            visited.add(current['_id'])

            parent = collection.find_one({"_id": current['parent_id']})
            if not parent:
                break

            parent_chain.append(parent)
            current = parent

        results.append({
            "target": node,
            "ancestors": parent_chain[::-1]
        })

    return results

# Connect to MongoDB
client = MongoClient('mongodb://localhost:27017/')
db = client['KB_PROPERTY_LAW']
collection = db['legal_sections']

# Extract data
diem_db_data = extract_all_with_parents(collection)
diem_data = {}

for diem in diem_db_data:
    content = ""
    title = ""

    for r in diem["ancestors"]:
        title += r['title'] + " "
        if not (r['title'].strip().startswith("chương")
                or r['title'].strip().startswith("điều")
                or r['title'].strip().startswith("mục")
        ):
            content += (r.get('content') or "") + "\n"

    title += diem['target']["title"]
    content += diem['target'].get("content") or ""

    so_hieu = diem['target']['so_hieu']
    ID = diem['target']['_id']

    if not content.strip():
        continue

    diem_data[ID] = {
        "so_hieu": so_hieu,
        "title": title,
        "content": content
    }
print(len(diem_data))

9050


In [59]:
import random

if diem_data:
    random_id = random.choice(list(diem_data.keys()))
    print(random_id)
    print(diem_data[random_id])
else:
    print("No data found.")

694fb81eaedc69db48c76e24
{'so_hieu': '96/2019/NĐ-CP', 'title': 'điều 6 khoản 1 điểm b', 'content': 'Bộ Tài nguyên và Môi trường có trách nhiệm:\nTổ chức điều chỉnh khung giá đất theo quy định của Luật Đất đai và Nghị định số 44/2014/NĐ-CP.'}


## Create batch api file

In [63]:

from dotenv import load_dotenv
import os

# Load .env files
load_dotenv()
api_key = os.getenv("OPENAI_API_KEY")

def get_gpt_response(user_prompt, system_prompt, api_key, model="gpt-4o"):
    client = OpenAI(api_key=api_key)
    response = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ],
    )
    print(response.choices[0].message.content)
    return response.choices[0].message.content

In [2]:
def create_prompt(so_hieu, title, content):
    system_prompt_rewrite = """
Bạn là trợ lý AI Tiếng Việt chuyên nghiệp và trung thực.
Bạn là chuyên gia pháp luật Việt Nam, am hiểu các bộ luật, nghị định, và văn bản pháp luật.
Bạn là chuyên gia ngôn ngữ Việt Nam, biết viết câu chuẩn cấu trúc, chính xác, trang trọng, và đúng ngôn ngữ pháp lý.
Luôn trả lời chính xác, hữu ích, ngắn gọn và an toàn.
Không thay đổi ý nghĩa khi viết lại câu.
Luôn dùng ngôn ngữ chính xác như trong văn bản pháp luật, tránh ngôn ngữ thông thường hay không trang trọng.
Định nghĩa 'câu đơn': Một câu đơn là câu có một chủ ngữ (hoặc cụm chủ ngữ) và một vị ngữ (hoặc cụm vị ngữ), biểu đạt một ý trọn vẹn; câu có thể chứa thành tố phụ (tính từ, trạng từ, bổ ngữ) nhưng không được ghép bằng liên từ hoặc dấu câu như dấu ",", ";" để tạo hai hoặc nhiều mệnh đề độc lập.

Quy tắc bắt buộc:
1. Khi viết lại, chỉ trả về các câu đơn theo đúng định nghĩa trên; mỗi câu một dòng nếu có nhiều câu.
2. Được phép tái sử dụng các thành phần câu (chủ ngữ, cụm danh từ, đại từ, cụm tính từ, v.v.) từ vế trước hoặc từ phần khác của câu gốc để hoàn chỉnh vế thiếu, nhằm bảo toàn ý nghĩa sau khi tách.
3. Khi tái sử dụng, ưu tiên giữ nguyên** từ ngữ gốc; chỉ thực hiện điều chỉnh nhỏ cần thiết để tạo câu đơn ngữ pháp đúng, **không** thêm thông tin, suy đoán hay nội dung mới.
4. Tuyệt đối không kèm chú giải, giải thích, danh sách hay bất kỳ nội dung nào khác ngoài các câu viết lại.
5. Nếu câu gốc mơ hồ hoặc thiếu thông tin đến mức không thể tạo câu đơn hoàn chỉnh mà vẫn giữ nguyên ý, hãy yêu cầu thêm thông tin ngắn gọn.
Danh mục liên từ cần loại trừ khi viết câu đơn: và, hoặc, hoặc là, hay, hay là, nhưng, song, tuy nhiên, mà, còn, rồi.
Giữ nguyên thứ tự trước sau của các từ sau khi viết lại câu.
Sau khi viết lại câu không được thiếu từ danh từ nào trong câu gốc và phải độc lập không phụ thuộc vào câu trước đó.
"""


    user_prompt_rewrite = f"""
Ngữ cảnh: Bộ luật số {so_hieu} trong luật Việt Nam, {title}
Nhiệm vụ: Viết lại câu sau để hoàn chỉnh cấu trúc với đầy đủ chủ ngữ và vị ngữ, giữ nguyên ý nghĩa. Mỗi câu xuất ra phải là một câu đơn đầy đủ (một dòng một câu nếu có nhiều câu).
Câu cần viết lại: "{content}"
"""
    return system_prompt_rewrite, user_prompt_rewrite

In [3]:


def normalize_for_json(obj):
    if isinstance(obj, ObjectId):
        return str(obj)
    if isinstance(obj, dict):
        return {k: normalize_for_json(v) for k, v in obj.items()}
    if isinstance(obj, list):
        return [normalize_for_json(v) for v in obj]
    return obj

In [71]:
import tiktoken

encoding = tiktoken.get_encoding("cl100k_base")

def count_chat_tokens(messages):
    """
    Rough but safe token count for chat.completions
    """
    tokens = 0
    for msg in messages:
        tokens += 4  # role + formatting overhead
        tokens += len(encoding.encode(msg["content"]))
    tokens += 2  # assistant reply priming
    return tokens

In [72]:
import json
import tiktoken
from typing import List

MODEL_NAME = "gpt-4.1-mini"
MAX_FILE_TOKENS = 2_000_000
MAX_TASK_TOKENS = 500_000   # safety limit per task before splitting content
OUTPUT_PREFIX = "batch_property_law_v1_part"

def normalize_for_json(obj):
    if isinstance(obj, dict):
        return {k: normalize_for_json(v) for k, v in obj.items()}
    elif isinstance(obj, list):
        return [normalize_for_json(v) for v in obj]
    elif isinstance(obj, tuple):
        return list(obj)
    return obj


def split_text_by_tokens(text: str, max_tokens: int) -> List[str]:
    """
    Split long content into token-safe chunks
    """
    tokens = encoding.encode(text)
    chunks = []

    for i in range(0, len(tokens), max_tokens):
        chunk_tokens = tokens[i:i + max_tokens]
        chunks.append(encoding.decode(chunk_tokens))

    return chunks

tasks = []
for key, value in diem_data.items():
    so_hieu = value["so_hieu"]
    title = value["title"]
    content = clean_text(value["content"])

    # split oversized content first
    content_chunks = split_text_by_tokens(content, MAX_TASK_TOKENS)

    for idx, chunk in enumerate(content_chunks, start=1):
        system_prompt, user_prompt = create_prompt(
            so_hieu,
            f"{title} (part {idx}/{len(content_chunks)})",
            chunk,
        )

        task = {
            "custom_id": key,
            "method": "POST",
            "url": "/v1/chat/completions",
            "body": {
                "model": MODEL_NAME,
                "messages": [
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": user_prompt},
                ],
            },
        }

        tokens = count_chat_tokens(task["body"]["messages"])

        if tokens > MAX_FILE_TOKENS:
            raise ValueError(
                f"Task {task['custom_id']} exceeds 2M tokens alone ({tokens})"
            )
        task["_token_count"] = tokens
        tasks.append(task)


files = []
current_file = []
current_tokens = 0

for task in tasks:
    if current_tokens + task["_token_count"] > MAX_FILE_TOKENS:
        files.append(current_file)
        current_file = []
        current_tokens = 0

    current_file.append(task)
    current_tokens += task["_token_count"]

if current_file:
    files.append(current_file)


for idx, file_tasks in enumerate(files, start=1):
    filename = f"{OUTPUT_PREFIX}{idx}.jsonl"

    with open(filename, "w", encoding="utf-8") as f:
        for task in file_tasks:
            task_copy = dict(task)
            task_copy.pop("_token_count", None)
            task_copy = normalize_for_json(task_copy)
            f.write(json.dumps(task_copy, ensure_ascii=False) + "\n")

    total_tokens = sum(t["_token_count"] for t in file_tasks)

    print(
        f"✔ {filename} | "
        f"{len(file_tasks)} tasks | "
        f"{total_tokens:,} tokens"
    )

print("\n===== SUMMARY =====")
print(f"Total tasks: {len(tasks)}")
print(f"Total files: {len(files)}")
print(f"Total tokens: {sum(t['_token_count'] for t in tasks):,}")

✔ batch_property_law_v1_part1.jsonl | 1689 tasks | 1,998,867 tokens
✔ batch_property_law_v1_part2.jsonl | 1698 tasks | 1,999,130 tokens
✔ batch_property_law_v1_part3.jsonl | 1626 tasks | 1,999,475 tokens
✔ batch_property_law_v1_part4.jsonl | 1618 tasks | 1,999,265 tokens
✔ batch_property_law_v1_part5.jsonl | 1627 tasks | 1,999,836 tokens
✔ batch_property_law_v1_part6.jsonl | 792 tasks | 1,051,719 tokens

===== SUMMARY =====
Total tasks: 9050
Total files: 6
Total tokens: 11,048,292


In [12]:
from openai import OpenAI
from dotenv import load_dotenv
import os

load_dotenv()
api_key = os.getenv("OPENAI_API_KEY")

client = OpenAI(api_key=api_key)
batch_file = client.files.create(
    file=open("batch_property_law_v1_part6.jsonl", "rb"),
    purpose="batch"
)

batch_job = client.batches.create(
    input_file_id=batch_file.id,
    endpoint="/v1/chat/completions",
    completion_window="24h"
)

In [3]:
import json

with open("batch_68f0720bbe3081908aa61019a6d518fe_output.jsonl", "r", encoding="utf-8") as f:
    for line in f:
        data = json.loads(line)
        if data.get("response", {}).get("status_code") != 200:
            continue
        content = data["response"]["body"]["choices"][0]["message"]["content"]
        print("────────────────────────────")
        print(data["custom_id"])
        print(content)

Việc quản lý các hạng mục công trình dưới đây được thực hiện theo quy định của thông tư này và quy định của pháp luật có liên quan.  
Việc khai thác các hạng mục công trình dưới đây được thực hiện theo quy định của thông tư này và quy định của pháp luật có liên quan.  
Việc bảo trì các hạng mục công trình dưới đây được thực hiện theo quy định của thông tư này và quy định của pháp luật có liên quan.  
Việc quản lý công trình dân dụng được thực hiện theo quy định của pháp luật về công trình dân dụng.  
Việc quản lý công trình hạ tầng kỹ thuật đô thị được thực hiện theo quy định của pháp luật về công trình dân dụng.  
Việc quản lý công trình công nghiệp vật liệu xây dựng được thực hiện theo quy định của pháp luật.  
Việc quản lý công trình hạ tầng kỹ thuật đô thị được thực hiện theo quy định của pháp luật.


## Lưu kết quả xử lý vào MongoDB

In [1]:
from src.triplet_extraction.src import init_mongo
from pathlib import Path
import json
from bson import ObjectId
from tqdm import tqdm

result_path = Path(r"/src/triplet_extraction\data\luat_dat_dai\simplify_sentence_data")

mongo_client = init_mongo()
if not mongo_client:
    print("Failed to connect to MongoDB. Exiting.")
    exit(1)

db = mongo_client["KB_PROPERTY_LAW"]

section_collection = db["legal_sections"]
collection = db["processed_legal_sections"]

files = list(result_path.glob("batch_*_output.jsonl"))

total_sentences = 0
missing_sections = 0

# 🔹 File-level progress bar
for file in tqdm(files, desc="Processing files", unit="file"):
    with open(file, "r", encoding="utf-8") as f:
        lines = list(f)

    # 🔹 Line-level progress bar
    for line in tqdm(lines, desc=f"{file.name}", unit="line", leave=False):
        data = json.loads(line)

        if data.get("response", {}).get("status_code") != 200:
            continue

        content = data["response"]["body"]["choices"][0]["message"]["content"].strip()
        split_content = content.split("\n")

        section_id_str = data["custom_id"].split("_part")[0]
        section_oid = ObjectId(section_id_str)

        section_doc = section_collection.find_one(
            {"_id": section_oid},
            {"so_hieu": 1}
        )

        if not section_doc:
            missing_sections += 1
            continue

        so_hieu = section_doc["so_hieu"]

        sequence = 1
        for c in split_content:
            c = c.strip()
            if not c:
                continue

            collection.update_one(
                {
                    "section_id": section_oid,
                    "sequence": sequence
                },
                {
                    "$setOnInsert": {
                        "content": c,
                        "so_hieu": so_hieu
                    }
                },
                upsert=True
            )

            total_sentences += 1
            sequence += 1

print(f"Sentences inserted : {total_sentences}")
print(f"Missing sections   : {missing_sections}")

You successfully connected to MongoDB!


Processing files:   0%|          | 0/6 [00:00<?, ?file/s]
batch_694fd2dc0794819098600c1480529b5d_output.jsonl:   0%|          | 0/1689 [00:00<?, ?line/s]
batch_694fd2dc0794819098600c1480529b5d_output.jsonl:   4%|▎         | 62/1689 [00:00<00:02, 608.80line/s]
batch_694fd2dc0794819098600c1480529b5d_output.jsonl:   7%|▋         | 123/1689 [00:00<00:03, 431.02line/s]
batch_694fd2dc0794819098600c1480529b5d_output.jsonl:  10%|█         | 170/1689 [00:00<00:04, 378.55line/s]
batch_694fd2dc0794819098600c1480529b5d_output.jsonl:  12%|█▏        | 210/1689 [00:00<00:04, 368.40line/s]
batch_694fd2dc0794819098600c1480529b5d_output.jsonl:  15%|█▍        | 248/1689 [00:00<00:04, 354.89line/s]
batch_694fd2dc0794819098600c1480529b5d_output.jsonl:  17%|█▋        | 284/1689 [00:00<00:04, 318.99line/s]
batch_694fd2dc0794819098600c1480529b5d_output.jsonl:  19%|█▉        | 317/1689 [00:00<00:04, 317.04line/s]
batch_694fd2dc0794819098600c1480529b5d_output.jsonl:  21%|██        | 350/1689 [00:01<00:04, 292.2

Sentences inserted : 33736
Missing sections   : 0


## Lưu kết quả xử lý vào SQLite

In [2]:
import json
from tqdm import tqdm
import sqlite3

conn = sqlite3.connect("process_law.db")
cursor = conn.cursor()

cursor.execute("""
CREATE TABLE IF NOT EXISTS laws (
    id TEXT,
    sequence INTEGER,
    content TEXT,
    so_hieu TEXT NOT NULL,
    PRIMARY KEY (id, sequence)
);
""")
conn.commit()


In [3]:
# Paths
db_path = r"/src/triplet_extraction/batch_preprocessing/process_law.db"
batch_result_path = r"/src/triplet_extraction\batch_68f0720bbe3081908aa61019a6d518fe_output.jsonl"

# Initialize database connection
conn = sqlite3.connect(db_path)

cursor = conn.cursor()
cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
tables = cursor.fetchall()
print("Các bảng trong law.db:", tables)

Các bảng trong law.db: [('laws',)]


In [4]:
import sqlite3
gtvt_conn = sqlite3.connect(r"/src/triplet_extraction/batch_preprocessing/GTVT_law.db")
gtvt_cursor = gtvt_conn.cursor()

In [5]:
def extract_from_sqlite(cursor, id: int, include_parent=False):
    cursor.execute("SELECT * FROM laws WHERE id = ?", (id,))
    rows = cursor.fetchall()

    columns = [col[0] for col in cursor.description]
    results = [dict(zip(columns, row)) for row in rows]

    if include_parent:
        all_nodes = []
        for r in results:
            current = r
            while current['parent_id'] is not None:
                cursor.execute("SELECT * FROM laws WHERE id = ?", (current['parent_id'],))
                parent = cursor.fetchone()
                if parent is None:
                    break
                parent_dict = dict(zip(columns, parent))
                all_nodes.append(parent_dict)
                current = parent_dict
        results.extend(all_nodes)

    return results[::-1]

In [6]:
with open(batch_result_path, "r", encoding="utf-8") as f:
    lines = f.readlines()

inserted = 0

for line in tqdm(lines, desc="Inserting JSON results", unit="entry"):
    data = json.loads(line)
    if data.get("response", {}).get("status_code") != 200:
        continue

    content = data["response"]["body"]["choices"][0]["message"]["content"]
    content = content.strip()
    split_content = content.replace("\n", "").split(".")
    law_id = data["custom_id"]

    # Fetch so_hieu from original DB if needed
    law_sentence = extract_from_sqlite(gtvt_cursor, law_id)
    so_hieu = law_sentence[0]["so_hieu"] if law_sentence else "unknown"
    sequence = 1
    for c in split_content:
        c = c.strip()
        if not c:
            continue
        cursor.execute("""
            INSERT OR REPLACE INTO laws (id, sequence, content, so_hieu)
            VALUES (?, ?, ?, ?)
        """, (law_id, sequence, c, so_hieu))
        inserted += 1
        sequence += 1

conn.commit()
print(f"Inserted {inserted} records into 'laws' table")

Inserting JSON results: 100%|██████████| 943/943 [00:00<00:00, 22056.20entry/s]

Inserted 4424 records into 'laws' table


In [7]:
law_sentence = extract_from_sqlite(gtvt_cursor, "3f2f65567a03467937fa6b9f291607028d46f8809afdcbf2303e905cb8ed8ab8")
print(law_sentence)

[{'id': '3f2f65567a03467937fa6b9f291607028d46f8809afdcbf2303e905cb8ed8ab8', 'title': 'Điểm e', 'content': 'Tổng hợp chi phí vận hành công trình trạm trong 1 ca (hoặc 1 ngày) làm việc BẢNG 2.2.A - TỔNG HỢP CHI PHÍ NHÂN CÔNG, MÁY THI CÔNG, VẬT TƯ VẬT LIỆU TRONG CHI PHÍ TRỰC TIẾP VẬN HÀNH CÔNG TRÌNH TRẠM TRONG 1 ĐƠN VỊ THỜI GIAN (CA, NGÀY) STT Mã hiệu Nội dung Đơn vị Khối lượng Giá Thành tiền [1] [2] [3] [4] [5] [6] [7] = [5]x[6] I Nhân công', 'parent_id': '63cbb6d6c4f372508b3868b7ac7e5cf310b41a3c4fa8d4ca4c86b32351ca73a0', 'so_hieu': '41/2021/TT-BGTVT'}]
